<a href="https://colab.research.google.com/github/moniii25007-maker/multiple-linear-regression/blob/main/moniga_HW_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Business Problem :

A real-estate company in Ames wants to identify different housing market segments based on property location, sale price, and housing characteristics. Manually analyzing thousands of properties is time-consuming and inefficient. By applying K-Means Clustering, the company aims to automatically group similar houses, helping management make better decisions for investment, pricing strategies, and targeted marketin

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [2]:
url = "https://gist.githubusercontent.com/manifoldhiker/51e93463aedb603fa7034bf3f430daaf/raw/ames.csv"
df = pd.read_csv(url)

In [3]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Fence,Misc_Feature,Misc_Val,Mo_Sold,Year_Sold,Sale_Type,Sale_Condition,Sale_Price,Longitude,Latitude
0,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,141,31770,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,...,No_Fence,NaN,0,5,2010,WD,Normal,215000,-93.619754,42.054035
1,One_Story_1946_and_Newer_All_Styles,Residential_High_Density,80,11622,Pave,No_Alley_Access,Regular,Lvl,AllPub,Inside,...,Minimum_Privacy,NaN,0,6,2010,WD,Normal,105000,-93.619756,42.053014
2,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,81,14267,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,...,No_Fence,Gar2,12500,6,2010,WD,Normal,172000,-93.619387,42.052659
3,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,93,11160,Pave,No_Alley_Access,Regular,Lvl,AllPub,Corner,...,No_Fence,NaN,0,4,2010,WD,Normal,244000,-93.617320,42.051245
4,Two_Story_1946_and_Newer,Residential_Low_Density,74,13830,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Inside,...,Minimum_Privacy,NaN,0,3,2010,WD,Normal,189900,-93.638933,42.060899


In [4]:
df.shape

(2930, 81)

In [5]:
df.columns.tolist()

['MS_SubClass',
 'MS_Zoning',
 'Lot_Frontage',
 'Lot_Area',
 'Street',
 'Alley',
 'Lot_Shape',
 'Land_Contour',
 'Utilities',
 'Lot_Config',
 'Land_Slope',
 'Neighborhood',
 'Condition_1',
 'Condition_2',
 'Bldg_Type',
 'House_Style',
 'Overall_Qual',
 'Overall_Cond',
 'Year_Built',
 'Year_Remod_Add',
 'Roof_Style',
 'Roof_Matl',
 'Exterior_1st',
 'Exterior_2nd',
 'Mas_Vnr_Type',
 'Mas_Vnr_Area',
 'Exter_Qual',
 'Exter_Cond',
 'Foundation',
 'Bsmt_Qual',
 'Bsmt_Cond',
 'Bsmt_Exposure',
 'BsmtFin_Type_1',
 'BsmtFin_SF_1',
 'BsmtFin_Type_2',
 'BsmtFin_SF_2',
 'Bsmt_Unf_SF',
 'Total_Bsmt_SF',
 'Heating',
 'Heating_QC',
 'Central_Air',
 'Electrical',
 'First_Flr_SF',
 'Second_Flr_SF',
 'Low_Qual_Fin_SF',
 'Gr_Liv_Area',
 'Bsmt_Full_Bath',
 'Bsmt_Half_Bath',
 'Full_Bath',
 'Half_Bath',
 'Bedroom_AbvGr',
 'Kitchen_AbvGr',
 'Kitchen_Qual',
 'TotRms_AbvGrd',
 'Functional',
 'Fireplaces',
 'Fireplace_Qu',
 'Garage_Type',
 'Garage_Finish',
 'Garage_Cars',
 'Garage_Area',
 'Garage_Qual',
 'Garage

In [6]:
df.dtypes

,0
MS_SubClass,object
MS_Zoning,object
Lot_Frontage,int64
Lot_Area,int64
Street,object
...,...
Sale_Type,object
Sale_Condition,object
Sale_Price,int64
Longitude,float64


In [7]:
df.isnull().sum()

,0
MS_SubClass,0
MS_Zoning,0
Lot_Frontage,0
Lot_Area,0
Street,0
...,...
Sale_Type,0
Sale_Condition,0
Sale_Price,0
Longitude,0


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.describe

<bound method NDFrame.describe of                               MS_SubClass                 MS_Zoning  \
0     One_Story_1946_and_Newer_All_Styles   Residential_Low_Density   
1     One_Story_1946_and_Newer_All_Styles  Residential_High_Density   
2     One_Story_1946_and_Newer_All_Styles   Residential_Low_Density   
3     One_Story_1946_and_Newer_All_Styles   Residential_Low_Density   
4                Two_Story_1946_and_Newer   Residential_Low_Density   
...                                   ...                       ...   
2925                  Split_or_Multilevel   Residential_Low_Density   
2926  One_Story_1946_and_Newer_All_Styles   Residential_Low_Density   
2927                          Split_Foyer   Residential_Low_Density   
2928  One_Story_1946_and_Newer_All_Styles   Residential_Low_Density   
2929             Two_Story_1946_and_Newer   Residential_Low_Density   

      Lot_Frontage  Lot_Area Street            Alley           Lot_Shape  \
0              141     31770   Pave  No_Alley_Access  Slightly_Irregular   
1               80     11622   Pave  No_Alley_Access             Regular   
2               81     14267   Pave  No_Alley_Access  Slightly_Irregular   
3               93     11160   Pave  No_Alley_Access             Regular   
4               74     13830   Pave  No_Alley_Access  Slightly_Irregular   
...            ...       ...    ...              ...                 ...   
2925            37      7937   Pave  No_Alley_Access  Slightly_Irregular   
2926             0      8885   Pave  No_Alley_Access  Slightly_Irregular   
2927            62     10441   Pave  No_Alley_Access             Regular   
2928            77     10010   Pave  No_Alley_Access             Regular   
2929            74      9627   Pave  No_Alley_Access             Regular   

     Land_Contour Utilities Lot_Config  ...            Fence Misc_Feature  \
0             Lvl    AllPub     Corner  ...         No_Fence          NaN   
1             Lvl    AllPub     Inside  ...  Minimum_Privacy          NaN   
2             Lvl    AllPub     Corner  ...         No_Fence         Gar2   
3             Lvl    AllPub     Corner  ...         No_Fence          NaN   
4             Lvl    AllPub     Inside  ...  Minimum_Privacy          NaN   
...           ...       ...        ...  ...              ...          ...   
2925          Lvl    AllPub    CulDSac  ...     Good_Privacy          NaN   
2926          Low    AllPub     Inside  ...  Minimum_Privacy          NaN   
2927          Lvl    AllPub     Inside  ...  Minimum_Privacy         Shed   
2928          Lvl    AllPub     Inside  ...         No_Fence          NaN   
2929          Lvl    AllPub     Inside  ...         No_Fence          NaN   

     Misc_Val Mo_Sold Year_Sold Sale_Type Sale_Condition Sale_Price  \
0           0       5      2010       WD          Normal     215000   
1           0       6      2010       WD          Normal     105000   
2       12500       6      2010       WD          Normal     172000   
3           0       4      2010       WD          Normal     244000   
4           0       3      2010       WD          Normal     189900   
...       ...     ...       ...       ...            ...        ...   
2925        0       3      2006       WD          Normal     142500   
2926        0       6      2006       WD          Normal     131000   
2927      700       7      2006       WD          Normal     132000   
2928        0       4      2006       WD          Normal     170000   
2929        0      11      2006       WD          Normal     188000   

      Longitude   Latitude  
0    -93.619754  42.054035  
1    -93.619756  42.053014  
2    -93.619387  42.052659  
3    -93.617320  42.051245  
4    -93.638933  42.060899  
...         ...        ...  
2925 -93.604776  41.988964  
2926 -93.602680  41.988314  
2927 -93.606847  41.986510  
2928 -93.600190  41.990921  
2929 -93.599996  41.989265  

[2930 rows x 81 columns]>

In [10]:
features = ['Lot_Frontage','Lot_Area','Sale_Price','Longitude','Latitude']
x = df[features]
x

,Lot_Frontage,Lot_Area,Sale_Price,Longitude,Latitude
0,141,31770,215000,-93.619754,42.054035
1,80,11622,105000,-93.619756,42.053014
2,81,14267,172000,-93.619387,42.052659
3,93,11160,244000,-93.617320,42.051245
4,74,13830,189900,-93.638933,42.060899
...,...,...,...,...,...
2925,37,7937,142500,-93.604776,41.988964
2926,0,8885,131000,-93.602680,41.988314
2927,62,10441,132000,-93.606847,41.986510
2928,77,10010,170000,-93.600190,41.990921


In [11]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_scaled
pd.DataFrame(x_scaled, columns=features)

,Lot_Frontage,Lot_Area,Sale_Price,Longitude,Latitude
0,2.488592,2.744381,0.428229,0.900671,1.062250
1,0.667355,0.187097,-0.948957,0.900593,1.006782
2,0.697212,0.522814,-0.110125,0.914942,0.987496
3,1.055488,0.128458,0.791305,0.995397,0.910677
4,0.488217,0.467348,0.113980,0.154266,1.435153
...,...,...,...,...,...
2925,-0.616467,-0.280621,-0.479462,1.483581,-2.472886
2926,-1.721152,-0.160296,-0.623440,1.565153,-2.508198
2927,0.129941,0.037199,-0.610920,1.402983,-2.606205
2928,0.577786,-0.017506,-0.135165,1.662058,-2.366567
